# Selbstlernskript: PatchTST mit Hugging Face

**Bearbeitungszeit:** ca. 15 Minuten
Das Skript basiert auf Vervollständigung von Code (Zellen mit # TODO:) und der Beantwortung von Reflexionsfragen

## Key Learnings

1. Warum zerlegt PatchTST Zeitreihen in **Patches**?
2. Was bedeutet **Channel Independence**?
3. Wie hängen `context_length`, `prediction_length`, `patch_length` und `patch_stride` zusammen?
4. Wie trainiert man PatchTST supervised mit HuggingFace?


## 0. Setup

Pakete installieren, falls noch nicht vorhanden:

```bash
pip install transformers torch pandas matplotlib
```


In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import PatchTSTConfig, PatchTSTForPrediction

torch.manual_seed(42)

print("PyTorch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

## 1. Daten laden

In [ ]:
# Wenn df bereits existiert, wird es genutzt. Sonst wird weather.csv geladen.
try:
    df
    print("Bestehendes df wird verwendet.")
except NameError:
    df = pd.read_csv("data\weather.csv")
    print("weatherHistory.csv wurde geladen.")

In [ ]:
# TODO: Nutze geeignete Funktionen, um dir einen Überblick über die vorhandenen Signale des DataFrames zu machen

In [ ]:
# Bestimme die Zeitspalte der Zeitserie:
time_col = "Formatted Date"

# TODO: Wähle nun die Features aus, die du vorhersagen möchtest:
target_cols = []

In [ ]:
# Vorverarbeitung

df = df[[time_col] + target_cols].dropna().copy() # Limitieren auf relevante Spalten und NaN-Werte entfernen
df[time_col] = pd.to_datetime(df[time_col], utc=True) # Zeitspalte ins DateTime-Format bringen
df = df.sort_values(time_col).reset_index(drop=True) # sortieren

df.head()

### Mini-Aufgabe 1

Visualisiere eine Variable. Was sehen Sie: Tageszyklus, Ausreißer, Trend?


In [ ]:
# TODO: Hier ist Platz für deinen Code

## 2. Dataset-Klasse

Jedes Sample besteht aus:

```text
x = context_length vergangene Zeitschritte
y = prediction_length zukünftige Zeitschritte
```

Die Tensorform ist:

```text
x: [context_length, num_channels]
y: [prediction_length, num_channels]
```


In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self,df,time_col,target_cols,context_length,prediction_length,scaler=None,fit_scaler=False):
        df = df.copy()
        df[time_col] = pd.to_datetime(df[time_col], utc=True)
        df = df.sort_values(time_col).reset_index(drop=True)

        self.context_length = context_length
        self.prediction_length = prediction_length

        values = df[target_cols].values.astype("float32")

        if scaler is None:
            scaler = StandardScaler()

        if fit_scaler:
            scaler.fit(values)

        self.scaler = scaler

        values = self.scaler.transform(values)

        self.data = torch.tensor(values, dtype=torch.float32)

    def __len__(self):
        return len(self.data) - self.context_length - self.prediction_length + 1

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.context_length]
        y = self.data[
            idx + self.context_length :
            idx + self.context_length + self.prediction_length
        ]
        return x, y

## 3. Experimente definieren

Es sollen PatchTST-relevante Größen getestet werden:

- `context_length`: Wie viel Vergangenheit sieht das Modell?
- `prediction_length`: Wie weit wird vorhergesagt?
- `patch_length`: Wie viele Werte bilden ein Patch?
- `patch_stride`: Wie stark überlappen Patches?

Die Wetterdaten liegen stündlich vor:

- `168` = 7 Tage Vergangenheit
- `336` = 14 Tage Vergangenheit
- `24` = 24 Stunden Vorhersage
- `48` = 48 Stunden Vorhersage


In [ ]:
# TODO: Wähle hier sinnvolle Ausprägungen, welche trainiert werden sollen:

# TODO: Länge des Rückblick- und Vorhersagefensters
context_lengths = []
prediction_lengths = []

# TODO: Patch und Stride
patch_length = 
patch_stride = 

# TODO: Anzahl der Eingabekanäle
num_channels = 

# TODO: Wähle einen sinnvollen Train/Test-Split
n = len(df)
train_end = 
val_end = 

# Split
train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

# Größen der Datensätze ausgeben
print(len(train_df), len(val_df), len(test_df))

### Mini-Aufgabe 2

Berechnen Sie gedanklich die Anzahl der Patches für `context_length=168`, `patch_length=16`, `patch_stride=8`:

\[
N = \left\lfloor\frac{L-P}{S}\right\rfloor + 1
\]


In [ ]:
def num_patches(context_length, patch_length, patch_stride):
    # TODO: Berechne hier die Anzahl der Patches
    return num_patches

for cl in context_lengths:
    print(
        f"context_length={cl}: "
        f"{num_patches(cl, patch_length, patch_stride)} Patches"
    )

## 4. DataLoader und Modellconfigs erzeugen

In [ ]:
batch_size = 32
dataloaders = {}
modelconfigs = {}

for context_len in context_lengths:
    for pred_len in prediction_lengths:
        train_dataset = TimeSeriesDataset(
            train_df,
            time_col,
            target_cols,
            context_len,
            pred_len,
            scaler=None,
            fit_scaler=True,
        )

        scaler = train_dataset.scaler

        val_dataset = TimeSeriesDataset(
            val_df,
            time_col,
            target_cols,
            context_len,
            pred_len,
            scaler=scaler,
            fit_scaler=False,
        )

        test_dataset = TimeSeriesDataset(
            test_df,
            time_col,
            target_cols,
            context_len,
            pred_len,
            scaler=scaler,
            fit_scaler=False,
        )

        dataloaders[(context_len, pred_len)] = {
            "train": DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True),
            "val": DataLoader(val_dataset, batch_size=batch_size, shuffle=False),
            "test": DataLoader(test_dataset, batch_size=batch_size, shuffle=False),
        }
        
        modelconfigs[(context_len, pred_len)] = PatchTSTConfig(
        # TODO: Konfiguriere hier dein PatchTST-Modell 
        )

x, y = next(iter(dataloaders[(168, 24)]["train"]))
print("x shape:", x.shape)
print("y shape:", y.shape)
print("x mean:", x.mean().item())
print("x std:", x.std().item())

## 5. Modelle trainieren


In [ ]:
# TODO: Wähle die Anzahl der zu trainierenden Epochen
num_epochs = 

learning_curves = {}
models = {}

total = len(context_lengths) * len(prediction_lengths)
counter = 1

for context_len in context_lengths:
    for pred_len in prediction_lengths:
        print(f"\nModell {counter}/{total}: CL={context_len}, PL={pred_len}")
        counter += 1

        model = PatchTSTForPrediction(modelconfigs[(context_len, pred_len)])
        optimizer = AdamW(model.parameters(), lr=1e-4)

        train_loader = dataloaders[(context_len, pred_len)]["train"]
        val_loader = dataloaders[(context_len, pred_len)]["val"]

        train_losses = []
        val_losses = []

        for epoch in range(num_epochs):
            model.train()
            train_loss = 0.0

            for x_batch, y_batch in train_loader:
                optimizer.zero_grad()

                outputs = model(
                    past_values=x_batch,
                    future_values=y_batch
                )

                loss = outputs.loss
                loss.backward()
                optimizer.step()

                train_loss += loss.item()

            train_loss /= len(train_loader)
            train_losses.append(train_loss)

            model.eval()
            val_loss = 0.0

            with torch.no_grad():
                for x_batch, y_batch in val_loader:
                    outputs = model(
                        past_values=x_batch,
                        future_values=y_batch
                    )
                    val_loss += outputs.loss.item()

            val_loss /= len(val_loader)
            val_losses.append(val_loss)

            print(
                f"Epoch {epoch+1}/{num_epochs} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Val Loss: {val_loss:.4f}"
            )

        learning_curves[(context_len, pred_len)] = {
            "train_loss": train_losses,
            "val_loss": val_losses,
        }
        models[(context_len, pred_len)] = model

        # Modellparameter und Lernkurven abspeichern
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "config": model.config,
            },
            f"trained_models_exercise\model_context_{context_len}_pred_{pred_len}.pt"
        )

# Lernkurven aller Trainingsläufe abspeichern
torch.save(learning_curves, "trained_models_exercise\learning_curves.pt")

In [ ]:
x_batch, y_batch = next(iter(dataloaders[(168, 24)]["test"]))

print(x_batch.mean())
print(x_batch.std())

print(y_batch.mean())
print(y_batch.std())

## 6. Learning Curves vergleichen

`num_epochs=1` --> nur 1 Punkt sichtbar. Für echte Kurven `num_epochs>1`


In [ ]:
def describe_structure(obj, indent=0):
    prefix = "    " * indent

    if isinstance(obj, dict):
        print(f"{prefix}dict ({len(obj)} keys)")
        for k, v in obj.items():
            print(f"{prefix}├── {k}")
            describe_structure(v, indent + 1)

    elif isinstance(obj, list):
        print(f"{prefix}list (len={len(obj)})")
        if obj:
            describe_structure(obj[0], indent + 1)

    else:
        print(f"{prefix}{type(obj).__name__}")

In [ ]:
# Lernkurven laden
learning_curves = torch.load("trained_models_exercise\learning_curves.pt")
# Strucktur des Dictionarys ausgeben:
describe_structure(learning_curves)

# TODO: Visualisiere deine Lernkurven: 
print(learning_curves)
plt.figure(figsize=(10, 5))

for (context_len, pred_len), curves in learning_curves.items():
    # TODO: Visualisiere deine Lernkurven: 

plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("PatchTST Learning Curves")
plt.legend()
plt.grid(True)
plt.show()

## 7. Vorhersagequalität auf Testdatensatz vergleichen

In [ ]:
import torch.nn.functional as F

# Evaluierung auf Test-Datensatz

test_mse = {}
test_mse_channels = {}

for context_len in context_lengths:
    for pred_len in prediction_lengths:

        checkpoint = torch.load(
            f"trained_models_exercise/model_context_{context_len}_pred_{pred_len}.pt",
            map_location="cpu",
            weights_only=False
        )

        model = PatchTSTForPrediction(checkpoint["config"])
        model.load_state_dict(checkpoint["model_state_dict"])
        model.eval()

        test_loader = dataloaders[(context_len, pred_len)]["test"]

        mse = 0.0

        # MSE pro Kanal sammeln
        channel_mse = torch.zeros(num_channels)

        with torch.no_grad():
            for x_batch, y_batch in test_loader:

                outputs = model(
                    past_values=x_batch,
                    future_values=y_batch
                )

                pred = outputs.prediction_outputs

                # Gesamt-MSE
                mse += outputs.loss.item()

                # MSE je Kanal
                for c in range(num_channels):
                    channel_mse[c] += F.mse_loss(
                        pred[:, :, c],
                        y_batch[:, :, c]
                    ).item()

        # Mittelwert über alle Test-Batches
        mse /= len(test_loader)
        channel_mse /= len(test_loader)

        test_mse[(context_len, pred_len)] = mse

        test_mse_channels[(context_len, pred_len)] = {
            target_cols[c]: channel_mse[c].item()
            for c in range(num_channels)
        }

        print(f"\nContext={context_len}, Prediction={pred_len}")
        print(f"Gesamt-MSE: {mse:.4f}")

        print("MSE je Kanal:")
        for name, value in test_mse_channels[(context_len, pred_len)].items():
            print(f"  {name:<25}: {value:.4f}")

In [ ]:
# Gesamt-MSE
labels = [
    f"CL={cl}\nPL={pl}"
    for cl, pl in test_mse.keys()
]

values = list(test_mse.values())

plt.figure(figsize=(8, 5))

bars = plt.bar(labels, values)

plt.ylabel("Test MSE")
plt.xlabel("Modell")
plt.title("Gesamt-MSE")
plt.grid(axis="y", alpha=0.3)

for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f"{bar.get_height():.2f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.show()


# MSE je Kanal
for channel in target_cols:

    values = [
        test_mse_channels[(cl, pl)][channel]
        for cl, pl in test_mse.keys()
    ]

    plt.figure(figsize=(8, 5))

    bars = plt.bar(labels, values)

    plt.ylabel("Test MSE")
    plt.xlabel("Modell")
    plt.title(f"MSE: {channel}")
    plt.grid(axis="y", alpha=0.3)

    for bar in bars:
        plt.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height(),
            f"{bar.get_height():.2f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

    plt.show()

### Mini-Aufgabe 3

Interpretieren, sofern ausreichend viele Epochen trainiert wurden:

1. Wird die Vorhersage schwieriger, wenn `prediction_length` größer wird?
2. Hilft eine längere `context_length`?
3. Siehst du Anzeichen für Underfitting oder Overfitting?

## 8. Eine Vorhersage visualisieren

Wählen Sie die beste oder interessanteste Konfiguration aus.


In [ ]:
selected_context = 336
selected_pred = 24
sample_idx = 123  # <-- beliebiges Test-Sample

model = models[(selected_context, selected_pred)]

test_dataset = dataloaders[(selected_context, selected_pred)]["test"].dataset

x, y = test_dataset[sample_idx]

# Batch-Dimension hinzufügen
x = x.unsqueeze(0)
y = y.unsqueeze(0)

model.eval()
with torch.no_grad():
    outputs = model(past_values=x)

pred = outputs.prediction_outputs

# Prüfen der Dimensionen
print("x shape:", x.shape)
print("y shape:", y.shape)
print("pred shape:", pred.shape)

for channel_idx, channel_name in enumerate(target_cols):

    plt.figure(figsize=(10, 4))

    # Vergangenheit
    plt.plot(
        range(selected_context),
        x[0, :, channel_idx].numpy(),
        label="History",
        color="tab:blue"
    )

    # Tatsächliche Zukunft
    plt.plot(
        range(selected_context, selected_context + selected_pred),
        y[0, :, channel_idx].numpy(),
        label="True",
        color="tab:green"
    )

    # Vorhersage
    plt.plot(
        range(selected_context, selected_context + selected_pred),
        pred[0, :, channel_idx].detach().numpy(),
        label="Prediction",
        color="tab:red",
        linestyle="--"
    )

    plt.axvline(
        selected_context - 1,
        color="black",
        linestyle=":",
        label="Forecast Start"
    )

    plt.title(
        f"{channel_name} | Sample={sample_idx} | "
        f"Context={selected_context}, Prediction={selected_pred}"
    )

    plt.xlabel("Time Step")
    plt.ylabel(channel_name)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

## 8. Abschlussreflexion

Beantworten Sie kurz:

1. Was ist der Unterschied zwischen `context_length` und `patch_length`?
2. Warum reduziert Patching den Rechenaufwand?
3. Warum ist Channel Independence bei vielen Sensor- oder Wetterdaten plausibel?
4. Bei welchen Daten wäre Channel Independence problematisch?
5. Welche Konfiguration hat am besten funktioniert?

## Merksatz

PatchTST unterscheidet sich vor allem dadurch, dass es **Zeitreihen in Patches tokenisiert** und **jeden Channel zunächst unabhängig verarbeitet**. Dadurch kann das Modell lange historische Fenster effizienter nutzen.


1. Was ist der Unterschied zwischen context_length und patch_length?
context_length gibt die Länge des gesamten Eingabefensters an, patch_length die Anzahl der Zeitpunkte, die zu einem Patch zusammengefasst werden.
2. Warum reduziert Patching den Rechenaufwand?
Mehrere Zeitpunkte werden zu einem Token zusammengefasst. Dadurch verarbeitet der Transformer weniger Tokens, wodurch insbesondere die Self-Attention effizienter wird.
3. Warum ist Channel Independence bei vielen Sensor- oder Wetterdaten plausibel?
Viele Variablen besitzen eine eigene zeitliche Dynamik und können unabhängig modelliert werden. Der gleiche Transformer lernt dabei allgemeine zeitliche Muster für alle Channels.
4. Bei welchen Daten wäre Channel Independence problematisch?
Bei stark gekoppelten Systemen, in denen sich Variablen gegenseitig beeinflussen, z. B. in chemischen Prozessen, Motorsteuerungen oder physikalischen Regelkreisen.
5. Welche Konfiguration hat am besten funktioniert?
Individuelle Antwort